# Ungraded Lab: Data Import Lab

## Overview 
Your team at EngageMetrics, a leading employee engagement analytics company, has just received datasets from multiple sources that need to be consolidated for an urgent executive presentation. The data includes employee feedback from regional offices in CSV format, educational achievement records in Excel spreadsheets, and compensation benchmarks from an external API. The leadership team needs insights on employee engagement trends by tomorrow morning.
<br>
In this hands-on lab, you'll learn how to efficiently import and process this scattered data using Python. You'll work with real EngageMetrics datasets containing employee insights, educational records, and external market data, mastering the essential data import techniques needed to handle various file formats and common data challenges.

<b>Pro tip:</b> Keep the lesson screencast handy while working through this lab. Even experienced developers regularly refer back to documentation and examples.

## Learning Outcomes 

By the end of this lab, you will be able to:
- Import data from CSV files using pandas
- Load data from Excel files with proper formatting
- Connect to and retrieve data from a REST API
- Handle common data import challenges (missing values, inconsistent formats)

## Dataset Information 
You'll work with two different EngageMetrics data sources and an external API from Wikipedia:

1. <b>employee_insights.csv:</b> Employee performance and demographic data
2. <b>education_data.xlsx:</b> Educational background information
3. <b>Wikipedia REST API:</b> Income statistics in the United States

## Activities

### Activity 1: Regional Feedback Data Import

Let's start with importing employee insights data from a CSV file. Import the quarterly employee engagement feedback data from all regions.  

<b>Step 1:</b> Import the required libraries first:

In [1]:
# Import required libraries
import pandas as pd
import requests
import json
from datetime import datetime

<b>Step 2:</b> Import and examine the CSV data:

In [2]:
# Import the employee insights data
employee_df = pd.read_csv('employee_insights.csv', na_values = ['','NA','null'])

# Display the first few rows
print("First few rows:")
employee_df.head()

First few rows:


,employee_id,age,salary,promotion_eligible,last_training_date,department,work_experience,projects_completed,hours_worked_weekly,work_mode,last_promotion_date,satisfaction_score,overtime_hours
0,E0001,54.0,NaN,NaN,15/08/2023,HR,NaN,14.0,NaN,remote work,2022-05-10,NaN,8.4
1,E0002,NaN,$64761,N,15/08/2023,NaN,1 years,NaN,53.3,HYBRID,05-10-2022,NaN,8.1
2,E0003,54.0,NaN,N,15/08/2023,Marketing,8,6.0,32.6,Hybrid,10/05/2022,10.0,5.2
3,E0004,NaN,NaN,No,NaN,NaN,16,1.0,37.8,Remote,05-10-2022,5.0,NaN
4,E0005,29.0,$61486,Y,15/08/2023,NaN,NaN,1.0,53.3,Hybrid,2022-05-10,NaN,0.3


<b>Tip:</b> Check the data types of your columns after import to ensure they're appropriate

<b>Step 3:</b> Handle data quality issues:
- Examine missing values
- Check for inconsistent date formats

In [3]:
# Check for missing values
print("Missing values:")
employee_df.isnull().sum()

Missing values:


employee_id             0
age                    56
salary                 37
promotion_eligible     16
last_training_date     29
department             15
work_experience        29
projects_completed     52
hours_worked_weekly    33
work_mode              16
last_promotion_date    26
satisfaction_score     39
overtime_hours         30
dtype: int64

In [4]:
# Identify inconsistent date formats
# 1. We define the colums we want to check / Definimos las columnas que queremos revisar
cols_dates = ['last_training_date', 'last_promotion_date']

# 2. We try the conversion (this creates a new Date DataFrame) / Intentamos la conversión (esto crea un nuevo DataFrame de fechas)
temp_dates = employee_df[cols_dates].apply(pd.to_datetime, dayfirst=True, errors='coerce')

# 3. We identify the rows where AT LEAST ONE of the two columns failed / Identificamos las filas donde AL MENOS UNA de las dos columnas falló
# .any(axis=1) looks to see if there is any True (NaT) in the row /.any(axis=1) busca si hay algún True (NaT) en la fila
inconsistent_rows = temp_dates.isna().any(axis=1) & employee_df[cols_dates].notna().any(axis=1)

inconsistent_df = employee_df[inconsistent_rows]

# 4. We show the results / Mostramos los resultados
print("Records with inconsistent date formats:")
if not inconsistent_df.empty:
    print(inconsistent_df[cols_dates])
else:
    print("No inconsistencies found!")

print(f"\nTotal rows with problems: {len(inconsistent_df)}")

Records with inconsistent date formats:
   last_training_date last_promotion_date
1          15/08/2023          05-10-2022
2          15/08/2023          10/05/2022
3                 NaN          05-10-2022
5          15/08/2023          10/05/2022
6          2023-08-15          05-10-2022
..                ...                 ...
95         15/08/2023                 NaN
96         2023-08-15          05-10-2022
97         15/08/2023                 NaN
98         15/08/2023                 NaN
99         15/08/2023          05-10-2022

[88 rows x 2 columns]

Total rows with problems: 88


## Here I'll try to clean or normalize the date format / Voy a intentar limpiar o normalizar el formato de fecha

In [5]:
# 1. Define the columns / Definimos las columnas
cols_dates = ['last_training_date', 'last_promotion_date']

# 2. Convert both columns to the standard pandas format / Convertimos AMBAS columnas al formato estándar de Pandas
# dayfirst=True fix all cases as 05-10-2022
for col in cols_dates:
    employee_df[col] = pd.to_datetime(employee_df[col], dayfirst=True, errors='coerce')

# 3. We print how it goes / verifiquemos cómo van
print("Normalize data:")
print(employee_df[cols_dates].head(10))

# 4. Verify they are not any more 'objects' (strings) but 'datetime'
print("\n Data type verification:")
print(employee_df[cols_dates].dtypes)

Normalize data:
  last_training_date last_promotion_date
0         2023-08-15          2022-10-05
1         2023-08-15                 NaT
2         2023-08-15                 NaT
3                NaT                 NaT
4         2023-08-15          2022-10-05
5         2023-08-15                 NaT
6                NaT                 NaT
7         2023-08-15          2022-10-05
8                NaT                 NaT
9                NaT                 NaT

 Data type verification:
last_training_date     datetime64[us]
last_promotion_date    datetime64[us]
dtype: object


### Activity 2: Educational Records Import
Now let's work with the educational background data from an Excel file. Process the talent development team's educational background data.

<b>Step 1:</b> Load the excel file: 

In [6]:
# Import the education data
# Load Excel file / Carga del archivo Excel
education_df = pd.read_excel('education_data.xlsx')

# Display the first five rows / Muestra los primeros 5 registros (filas)
print("Education data preview:")
education_df.head(5)

Education data preview:


,ID,graduation_year,educational_background
0,E0001,2011,Psychology
1,E0002,1995,Architecture
2,E0003,2007,Business Administration
3,E0004,2000,Business Administration
4,E0005,1991,Medicine


<b>Step 2:</b> Data validation:
- Verify graduation years are in the correct format
- Check for any missing educational backgrounds

In [7]:
# Verify whether graduation years are in the correct format
# Validate year format
valid_years = education_df['graduation_year'].between(1960, 2026)
print("Invalid years:", education_df[~valid_years]['graduation_year'].unique())  
# Check if there are any missing educational backgrounds
missing_count = education_df['educational_background'].isna().sum()
print("Number of missing backgrounds:", missing_count)

if missing_count > 0:
    print("IDs with missing backgrounds:")
    print(education_df[education_df['educational_background'].isna()]['ID'])

Invalid years: []
Number of missing backgrounds: 0


<b>Tip:</b> Sometimes data might be clean and in correct format, but it's always good to verify before proceeding further

### Activity 3: Compensation Benchmark Data Import
Fetch the summary of the article using the Wikipedia REST API.
The summary can be found in the JSON response under the key extract. Print it to the console.

<b>Step 1: Make an API request:</b>

Use the requests library to send a GET request to the Wikipedia REST API.

In [8]:
import requests
# Make the API request
headers = {
    "User-Agent": "MedicorpDataBot/1.0 (contact: tu_usuario@example.com)"
}
url = "https://en.wikipedia.org/api/rest_v1/page/summary/Income_in_the_United_States"
response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    print("Successfully retrieved:", data['title'])
else:
    print(f"Error: {response.status_code}")

Successfully retrieved: Income in the United States


<b>Note:</b> Wikipedia requires a User-Agent header to allow your request. This is necessary to avoid 403 Forbidden errors, even if not explicitly required by the lab instructions.

In [9]:
# See what data Wikipedia gave us
print("Available information:", data.keys())

Available information: dict_keys(['type', 'title', 'displaytitle', 'namespace', 'wikibase_item', 'titles', 'pageid', 'thumbnail', 'originalimage', 'lang', 'dir', 'revision', 'tid', 'timestamp', 'description', 'description_source', 'content_urls', 'extract', 'extract_html'])


<b>Step 2: Extract and display the article summary</b>

Extract the article summary from the JSON response using the key extract.If the key doesn’t exist, show a default message like "Summary not available".Print the summary to the console.

In [10]:
# Extract and display the article summary / Extraer y mostrar el resumen del artículo
# Usamos .get() en lugar de ['extract'] para evitar errores si la clave no existe
summary = data.get('extract', 'Summary not found')

print(f"--- Wikipedia Summary for {data['title']} ---")
print(summary)

--- Wikipedia Summary for Income in the United States ---
Income in the United States is measured by the various federal agencies including the Internal Revenue Service, Bureau of Labor Statistics, US Department of Commerce, and the US Census Bureau. Additionally, various agencies, including the Congressional Budget Office compile reports on income statistics. The primary classifications are by household or individual. The top quintile in personal income in 2022 was $117,162. The differences between household and personal income are considerable, since 61% of households now have two or more income earners. Median personal income in 2020 was $56,287 for workers with full-time employment.



#### Test Your Work:
1. Verify that all three datasets are loaded successfully
2. Confirm the data types are correct in each dataset
3. Check that you can access specific columns and rows
4. Ensure API response contains expected fields

## Success Checklist
- Imported regional feedback data successfully
- Processed educational achievement records
- Retrieved current compensation benchmarks
- Handled missing survey responses appropriately
- Verified data quality across all sources
- Documented data anomalies for the analytics team

## Common Issues & Solutions 
- Problem: Regional feedback data shows inconsistent scoring
    - Solution: Apply standardization across different regional formats
- Problem: Training records show duplicate entries
    - Solution: Implement deduplication logic while preserving the most recent records
- Problem: Benchmark API returns incomplete data
    - Solution: Implement proper error handling and data validation checks

## Summary 
Congratulations! You've successfully learned to import data from multiple sources (CSV, Excel, and API) using Python, and tackled real-world data quality challenges like handling missing values and inconsistent formats. Through this lab, you've gained practical experience with essential data import techniques that data scientists use daily, setting a strong foundation for more advanced data analysis tasks.

### Key Points
- Different data sources require different import approaches
- Always validate data after import
- Handle missing or inconsistent data appropriately
- Document any data quality issues discovered

## Solution Code
Stuck on your code or want to check your solution? Here's a complete reference implementation to guide you. This represents just one effective approach—try solving independently first, then use this to overcome obstacles or compare techniques. The solution is provided to help you move forward and explore alternative approaches to achieve the same results. Happy coding!

###  Activity 1: Regional Feedback Data Import - Solution Code

In [11]:
# Step 2: Import and examine the CSV data
employee_df = pd.read_csv('employee_insights.csv')

# Display the first few rows
print("Employee Data : ")
print(employee_df.head())

# Step 3: Handle data quality issues
# Check for missing values
print(employee_df.isnull().sum())

# Check for inconsistent date formats
# Columns with date info
date_columns = ['last_training_date', 'last_promotion_date']

for col in date_columns:
    # Extract year from the date
    employee_df[col + '_year'] = pd.to_datetime(
        employee_df[col], errors='coerce', dayfirst=True
    ).dt.year
    
    # Validate year using between()
    valid_years = employee_df[col + '_year'].between(1900, pd.Timestamp.now().year)
    invalid_years = employee_df.loc[~valid_years, col + '_year']
    
    print(f"Inconsistent years in {col}:")
    print(invalid_years.tolist())

Employee Data : 
  employee_id   age  salary promotion_eligible last_training_date department  \
0       E0001  54.0     NaN                NaN         15/08/2023         HR   
1       E0002   NaN  $64761                  N         15/08/2023        NaN   
2       E0003  54.0     NaN                  N         15/08/2023  Marketing   
3       E0004   NaN     NaN                 No                NaN        NaN   
4       E0005  29.0  $61486                  Y         15/08/2023        NaN   

  work_experience  projects_completed  hours_worked_weekly    work_mode  \
0             NaN                14.0                  NaN  remote work   
1         1 years                 NaN                 53.3       HYBRID   
2               8                 6.0                 32.6       Hybrid   
3              16                 1.0                 37.8       Remote   
4             NaN                 1.0                 53.3       Hybrid   

  last_promotion_date  satisfaction_score  overtime

### Activity 2: Excel Data Import - Solution Code

In [12]:
# Step 1: Load the Excel file
education_data = pd.read_excel('education_data.xlsx')

# Display the first few rows
print("Education Data : ")
print(education_data.head())

# Step 2: Data validation
# Verify graduation years are in the correct format

valid_years = education_data['graduation_year'].between(1900, pd.Timestamp.now().year)
invalid_years = education_data.loc[~valid_years, 'graduation_year']

if invalid_years.empty:
    print("All graduation years are well formatted.")
else:
    print("Invalid graduation years:")
    print(invalid_years.tolist())
    
# Check is there are any missing educational backgrounds
missing_education = education_data[education_data['educational_background'].isnull()]
if missing_education.empty:
    print("No missing educational backgrounds.")
else:
    print("Missing educational backgrounds:")
    print(missing_education)

Education Data : 
      ID  graduation_year   educational_background
0  E0001             2011               Psychology
1  E0002             1995             Architecture
2  E0003             2007  Business Administration
3  E0004             2000  Business Administration
4  E0005             1991                 Medicine
All graduation years are well formatted.
No missing educational backgrounds.


### Activity 3: Compensation Benchmark Data Import  - Solution Code

In [13]:
import requests

# URL of the Wikipedia article summary
url = "https://en.wikipedia.org/api/rest_v1/page/summary/Income_in_the_United_States"

headers = {
    # Wikipedia API requires a User-Agent header to avoid 403 Forbidden errors
    # This identifies your request politely; not required by the lab instructions
    "User-Agent": "MyApp/1.0 (contact: youremail@example.com)"
}

# Make the API request
response = requests.get(url, headers=headers)

# Parse the JSON response
data = response.json()

# Extract and display the article summary
summary = data.get('extract', 'Summary not available')
print("Article summary:")
print(summary)

Article summary:
Income in the United States is measured by the various federal agencies including the Internal Revenue Service, Bureau of Labor Statistics, US Department of Commerce, and the US Census Bureau. Additionally, various agencies, including the Congressional Budget Office compile reports on income statistics. The primary classifications are by household or individual. The top quintile in personal income in 2022 was $117,162. The differences between household and personal income are considerable, since 61% of households now have two or more income earners. Median personal income in 2020 was $56,287 for workers with full-time employment.

